Title: Add_SMARD_to_other.ipynb

Purpose: Add the SMARD Data to the main dataset

Author: Onno Nennecke on 09.06.2025 Modified: 09.06.2025

Input data: 

    - This file lies here: 

Output data:

    - This file lies here: 

In [1]:
# Importing libraries
import xarray as xr
import numpy as np
import pandas as pd

In [2]:
path = '/climca/people/onennecke/model_output/bias_corrected_masked/winter_data/adjusted/final_model_output.nc'
files = [path]

ts_datasets = xr.open_dataset(path)
# ts_datasets = ts_datasets.where(ts_datasets.ESM_run != 'ERA5_hist_wwd', drop=True)
ts_datasets.load()

<xarray.Dataset> Size: 42MB
Dimensions:             (ESM_run: 101, time: 1820)
Coordinates:
  * time                (time) datetime64[ns] 15kB 2015-01-01T12:00:00 ... 20...
    gridtype            <U6 24B 'lonlat'
    country             float64 8B 9.0
    period              <U4 16B 'week'
    run                 (ESM_run) <U10 4kB 'r1i1p1f1' 'r4i1p1f1' ... 'r9i1p1f2'
    ESM                 (ESM_run) <U13 5kB 'ACCESS-CM2' ... 'UKESM1-0-LL'
  * ESM_run             (ESM_run) <U23 9kB 'ACCESS-CM2_r1i1p1f1' ... 'UKESM1-...
    winter_year         (time) int64 15kB 2014 2014 2014 2014 ... 2024 2024 2024
    day_of_winter       (time) int64 15kB 93 94 95 96 97 98 ... 88 89 90 91 92
    winter_season       (time) <U8 58kB '2014-093' '2014-094' ... '2024-092'
    old_time            (ESM_run, time) <U19 14MB '1420113600000000000' ... '...
Data variables: (12/19)
    temp                (ESM_run, time) float64 1MB 4.32 1.109 ... 5.41 5.41
    demand              (ESM_run, time) float64 1MB 1.466e+03 ... 1.453e+03
    sfcWind             (ESM_run, time) float64 1MB 8.034 6.675 ... 9.436 9.436
    rsds                (ESM_run, time) float64 1MB 20.05 30.42 ... 19.66 19.66
    tas                 (ESM_run, time) float64 1MB 3.572 0.6657 ... 5.796 5.796
    tasmax              (ESM_run, time) float64 1MB 6.133 2.853 ... 6.983 6.983
    ...                  ...
    wind_on_prod_adj    (ESM_run, time) float64 1MB 554.3 412.6 ... 713.1 713.1
    solar_prod_adj      (ESM_run, time) float64 1MB -4.14 10.36 ... -2.277
    total_prod_adj      (ESM_run, time) float64 1MB 675.1 524.9 ... 852.7 852.7
    total_prod_adj_sum  (ESM_run, time) float64 1MB 669.8 530.9 ... 830.4 830.4
    Netto_adjusted      (ESM_run, time) float64 1MB -796.6 -975.3 ... -622.3
    Residual_load_adj   (ESM_run, time) float64 1MB 796.6 975.3 ... 622.3 622.3
Attributes:
    crs:      4326

### Load SMARD data

In [3]:
SMARD_data_prod = pd.read_csv('/home/onennecke/SMARD_data/Realisierte_Erzeugung_201501010000_202505010000_Tag.csv', sep=';', decimal=',', thousands='.')
SMARD_data_demand = pd.read_csv('/home/onennecke/SMARD_data/Realisierter_Stromverbrauch_201501010000_202505010000_Tag.csv', sep=';', decimal=',', thousands='.')

In [4]:

date = pd.to_datetime(SMARD_data_prod['Datum von'].astype(str).str.zfill(8), format='%d%m%Y')
demand_sm = SMARD_data_demand['Netzlast [MWh] Berechnete Auflösungen']
wind_off_prod = SMARD_data_prod['Wind Offshore [MWh] Berechnete Auflösungen']
wind_on_prod = SMARD_data_prod['Wind Onshore [MWh] Berechnete Auflösungen']
solar_prod = SMARD_data_prod['Photovoltaik [MWh] Berechnete Auflösungen']
total_prod = wind_off_prod + wind_on_prod + solar_prod
residual_load = demand_sm - total_prod

df = pd.DataFrame({
    'date': date,
    'demand_SMARD': demand_sm / 1000,  # Convert to GWh
    'wind_offshore_SMARD': wind_off_prod / 1000,  # Convert to GWh
    'wind_onshore_SMARD': wind_on_prod / 1000,  # Convert to GWh
    'solar_SMARD': solar_prod / 1000,  # Convert to GWh
    'total_production_SMARD': total_prod / 1000,  # Convert to GWh
    'residual_load_SMARD': residual_load / 1000,  # Convert to GWh
})


In [5]:
# Cut data to only inclued ONDJFM
df = df[df['date'].dt.month.isin([1, 2, 3, 10, 11, 12])]
df = df[(df['date'] >= '2015-01-01') & (df['date'] <= '2024-12-31')]
# Exclude 29th February in leap years
df = df[~((df['date'].dt.month == 2) & (df['date'].dt.day == 29))]
df

,date,demand_SMARD,wind_offshore_SMARD,wind_onshore_SMARD,solar_SMARD,total_production_SMARD,residual_load_SMARD
0,2015-01-01,1096.85275,12.50850,298.79175,17.08025,328.38050,768.47225
1,2015-01-02,1288.91475,10.32925,591.62075,7.75900,609.70900,679.20575
2,2015-01-03,1213.30950,12.11950,457.04350,7.23475,476.39775,736.91175
3,2015-01-04,1177.89600,11.53850,379.02950,19.98250,410.55050,767.34550
4,2015-01-05,1425.92750,7.74550,219.62350,26.52225,253.89125,1172.03625
...,...,...,...,...,...,...,...
3648,2024-12-27,1155.15125,2.94125,46.42200,61.20575,110.56900,1044.58225
3649,2024-12-28,1150.65975,22.34775,31.44225,61.08850,114.87850,1035.78125
3650,2024-12-29,1179.16200,97.29825,257.39450,39.71900,394.41175,784.75025
3651,2024-12-30,1278.28500,61.18675,534.34725,22.92750,618.46150,659.82350


In [6]:
# Get existing time coordinate
time = ts_datasets.coords['time']

# Example new data values (use real data in practice)
SMARD_data = {
    'temp': (['ESM_run', 'time'], np.full((1, len(time)), np.nan)),
    'demand': (['ESM_run', 'time'], df['demand_SMARD'].values.reshape(1, -1)),
    'sfcWind': (['ESM_run', 'time'], np.full((1, len(time)), np.nan)),
    'rsds': (['ESM_run', 'time'], np.full((1, len(time)), np.nan)),
    'tas': (['ESM_run', 'time'], np.full((1, len(time)), np.nan)),
    'tasmax': (['ESM_run', 'time'], np.full((1, len(time)), np.nan)),
    'wind_off_prod': (['ESM_run', 'time'], df['wind_offshore_SMARD'].values.reshape(1, -1)),
    'wind_on_prod': (['ESM_run', 'time'], df['wind_onshore_SMARD'].values.reshape(1, -1)),
    'solar_prod': (['ESM_run', 'time'], df['solar_SMARD'].values.reshape(1, -1)),
    'total_prod': (['ESM_run', 'time'], df['total_production_SMARD'].values.reshape(1, -1)),
    'Netto': (['ESM_run', 'time'], np.full((1, len(time)), np.nan)),
    'Residual_load': (['ESM_run', 'time'], df['residual_load_SMARD'].values.reshape(1, -1)),
    'wind_off_prod_adj': (['ESM_run', 'time'], df['wind_offshore_SMARD'].values.reshape(1, -1)),
    'wind_on_prod_adj': (['ESM_run', 'time'], df['wind_onshore_SMARD'].values.reshape(1, -1)),
    'solar_prod_adj': (['ESM_run', 'time'], df['solar_SMARD'].values.reshape(1, -1)),
    'total_prod_adj': (['ESM_run', 'time'],  df['total_production_SMARD'].values.reshape(1, -1)),
    'total_prod_adj_sum': (['ESM_run', 'time'], df['total_production_SMARD'].values.reshape(1, -1)),
    'Netto_adjusted': (['ESM_run', 'time'], np.full((1, len(time)), np.nan)),
    'Residual_load_adj': (['ESM_run', 'time'], df['residual_load_SMARD'].values.reshape(1, -1))
}

# New coordinate values for the new ESM_run
SMARD_coords = {
    'ESM_run': ['SMARD_hist'],
    'run': (['ESM_run'], ['hist']),
    'ESM': (['ESM_run'], ['SMARD']),
    'old_time': (['ESM_run', 'time'], ts_datasets.coords['old_time'][0].values.reshape(1, -1))
}

# Fixed coords broadcasted from the original dataset
static_coords = {
    'time': time,
    'gridtype': ts_datasets.gridtype,
    'country': ts_datasets.country,
    'period': ts_datasets.period,
    'winter_year': ts_datasets.winter_year,
    'day_of_winter': ts_datasets.day_of_winter,
    'winter_season': ts_datasets.winter_season,
}

# Combine all into a SMARD dataset
SMARD_ds = xr.Dataset(data_vars=SMARD_data, coords={**SMARD_coords, **static_coords})


### Combine both datasets

In [7]:
combined_ds = xr.concat([ts_datasets, SMARD_ds], dim='ESM_run')
combined_ds

<xarray.Dataset> Size: 42MB
Dimensions:             (ESM_run: 102, time: 1820)
Coordinates:
  * time                (time) datetime64[ns] 15kB 2015-01-01T12:00:00 ... 20...
    gridtype            <U6 24B 'lonlat'
    country             float64 8B 9.0
    period              <U4 16B 'week'
    run                 (ESM_run) <U10 4kB 'r1i1p1f1' 'r4i1p1f1' ... 'hist'
    ESM                 (ESM_run) <U13 5kB 'ACCESS-CM2' 'ACCESS-CM2' ... 'SMARD'
  * ESM_run             (ESM_run) <U23 9kB 'ACCESS-CM2_r1i1p1f1' ... 'SMARD_h...
    winter_year         (time) int64 15kB 2014 2014 2014 2014 ... 2024 2024 2024
    day_of_winter       (time) int64 15kB 93 94 95 96 97 98 ... 88 89 90 91 92
    winter_season       (time) <U8 58kB '2014-093' '2014-094' ... '2024-092'
    old_time            (ESM_run, time) <U19 14MB '1420113600000000000' ... '...
Data variables: (12/19)
    temp                (ESM_run, time) float64 1MB 4.32 1.109 ... nan nan
    demand              (ESM_run, time) float64 1MB 1.466e+03 ... 1.204e+03
    sfcWind             (ESM_run, time) float64 1MB 8.034 6.675 ... nan nan
    rsds                (ESM_run, time) float64 1MB 20.05 30.42 ... nan nan
    tas                 (ESM_run, time) float64 1MB 3.572 0.6657 ... nan nan
    tasmax              (ESM_run, time) float64 1MB 6.133 2.853 ... nan nan
    ...                  ...
    wind_on_prod_adj    (ESM_run, time) float64 1MB 554.3 412.6 ... 534.3 529.0
    solar_prod_adj      (ESM_run, time) float64 1MB -4.14 10.36 ... 22.93 49.09
    total_prod_adj      (ESM_run, time) float64 1MB 675.1 524.9 ... 618.5 647.4
    total_prod_adj_sum  (ESM_run, time) float64 1MB 669.8 530.9 ... 618.5 647.4
    Netto_adjusted      (ESM_run, time) float64 1MB -796.6 -975.3 ... nan nan
    Residual_load_adj   (ESM_run, time) float64 1MB 796.6 975.3 ... 659.8 556.8
Attributes:
    crs:      4326

In [8]:
# Save the combined dataset
output_path = '/climca/people/onennecke/model_output/bias_corrected_masked/winter_data/adjusted/final_model_output_with_SMARD.nc'
combined_ds.to_netcdf(output_path)